# CS156 Session 7: Neural Networks

Describing and drawing feed-forward networks, writing the forward pass as matrix multiplications, and a perceptron from scratch.

My written answers are in `session7_workbook.md`.

Notes on this notebook:
- The two diagram cells need `mlp_architecture_diagram.png` and `perceptron_architecture_diagram.png` in the same folder as the notebook.
- The Keras cell, the sonar cell, and the FashionMNIST cell were **not run** here. They need TensorFlow, `sonar.all-data.csv`, and `fashion-mnist_train.csv`, none of which I have. Run them where those are available.

## Q1: The Keras network on MNIST

Written answers to the eleven questions (data, task, layers, units, activations, weights, output, performance) are in `session7_workbook.md`.

Issues in this code, covered in my answer to question 11:
- `BinaryCrossentropy` is the wrong loss for a 10-way, mutually exclusive classification. `CategoricalCrossentropy` with a softmax output would fit.
- `x_val` and `y_val` are never normalized or one-hot encoded.
- `model.fit` is called twice, so the model trains for 20 epochs in total.

In [8]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.losses import BinaryCrossentropy

(x_train, y_train), (x_val, y_val) = mnist.load_data()

x_train = x_train.astype('float32') / 255
x_val = x_val.astype('float32') / 255
y_train = to_categorical(y_train)
y_val = to_categorical(y_val)

model = tf.keras.Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='sigmoid'),
    Dense(64, activation='sigmoid'),
    Dense(10)
])

model.compile(optimizer='SGD',
              loss=BinaryCrossentropy(),
              metrics=['accuracy'])
print(x_train.shape, y_train.shape, x_val.shape, y_val.shape)
# (60000, 28, 28) (60000, 10) (10000, 28, 28) (10000, 10)
model.fit(x_train, y_train, epochs=10)

model.fit(x_train, y_train, epochs=10, validation_data=(x_val, y_val))

(60000, 28, 28) (60000, 10) (10000, 28, 28) (10000, 10)
Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 548us/step - accuracy: 0.1037 - loss: 2.3377
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 582us/step - accuracy: 0.1044 - loss: 2.3147
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 561us/step - accuracy: 0.1003 - loss: 3.3547
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 550us/step - accuracy: 0.1095 - loss: 5.4687
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 571us/step - accuracy: 0.1782 - loss: 6.6639
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 584us/step - accuracy: 0.1766 - loss: 6.6704
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 572us/step - accuracy: 0.1801 - loss: 6.6677
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 591us/step - accuracy: 0.1806 - loss: 6.6660
Epoch 9/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 574us/step - accuracy: 0.1810 - loss: 6.6580
Epoch 10/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 568us/step - accuracy: 0.1771 - loss: 6.6732
Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1

## Q2: Diagram and forward pass for the MNIST network

Each layer is a rectangle. Every arrow is fully connected (each unit connects to every unit in the next layer) and is labeled with the matrix multiplication it performs. The final layer has no activation, so its output is raw logits.

```
z1 = sigmoid(W1 x + b1)       W1: 128 x 784
z2 = sigmoid(W2 z1 + b2)      W2: 64 x 128
y_hat = W3 z2 + b3            W3: 10 x 64

y_hat = W3 sigmoid(W2 sigmoid(W1 x + b1) + b2) + b3
```

## Q3: A perceptron from scratch (Brownlee)

The perceptron helper functions from the workbook, defined once here. They work on any list of rows whose last entry is a 0/1 label.

In [11]:
#retrieved from https://machinelearningmastery.com/implement-perceptron-algorithm-scratch-python/
#Jason Brownlee

from random import seed, randrange
from csv import reader

# Load a CSV file
def load_csv(filename):
    dataset = list()
    with open(filename, 'r') as file:
        csv_reader = reader(file)
        for row in csv_reader:
            if not row:
                continue
            dataset.append(row)
    return dataset

# Convert string column to float
def str_column_to_float(dataset, column):
    for row in dataset:
        row[column] = float(row[column].strip())

# Convert string column to integer
def str_column_to_int(dataset, column):
    class_values = [row[column] for row in dataset]
    unique = set(class_values)
    lookup = dict()
    for i, value in enumerate(unique):
        lookup[value] = i
    for row in dataset:
        row[column] = lookup[row[column]]
    return lookup

# Split a dataset into k folds
def cross_validation_split(dataset, n_folds):
    dataset_split = list()
    dataset_copy = list(dataset)
    fold_size = int(len(dataset) / n_folds)
    for i in range(n_folds):
        fold = list()
        while len(fold) < fold_size:
            index = randrange(len(dataset_copy))
            fold.append(dataset_copy.pop(index))
        dataset_split.append(fold)
    return dataset_split

# Calculate accuracy percentage
def accuracy_metric(actual, predicted):
    correct = 0
    for i in range(len(actual)):
        if actual[i] == predicted[i]:
            correct += 1
    return correct / float(len(actual)) * 100.0

# Evaluate an algorithm using a cross validation split
def evaluate_algorithm(dataset, algorithm, n_folds, *args):
    folds = cross_validation_split(dataset, n_folds)
    scores = list()
    for fold in folds:
        train_set = list(folds)
        train_set.remove(fold)
        train_set = sum(train_set, [])
        test_set = list()
        for row in fold:
            row_copy = list(row)
            test_set.append(row_copy)
            row_copy[-1] = None
        predicted = algorithm(train_set, test_set, *args)
        actual = [row[-1] for row in fold]
        accuracy = accuracy_metric(actual, predicted)
        scores.append(accuracy)
    return scores

# Make a prediction with weights
def predict(row, weights):
    activation = weights[0]
    for i in range(len(row)-1):
        activation += weights[i + 1] * row[i]
    return 1.0 if activation >= 0.0 else 0.0

# Estimate Perceptron weights using stochastic gradient descent
def train_weights(train, l_rate, n_epoch):
    weights = [0.0 for i in range(len(train[0]))]
    for epoch in range(n_epoch):
        for row in train:
            prediction = predict(row, weights)
            error = row[-1] - prediction
            weights[0] = weights[0] + l_rate * error
            for i in range(len(row)-1):
                weights[i + 1] = weights[i + 1] + l_rate * error * row[i]
    return weights

# Perceptron Algorithm With Stochastic Gradient Descent
def perceptron(train, test, l_rate, n_epoch):
    predictions = list()
    weights = train_weights(train, l_rate, n_epoch)
    for row in test:
        prediction = predict(row, weights)
        predictions.append(prediction)
    return(predictions)

Running it on the sonar data. This is the original code from the workbook. It needs `sonar.all-data.csv`, which the workbook didn't have, so it raised `FileNotFoundError` there.

In [ ]:
# Test the Perceptron algorithm on the sonar dataset
seed(1)
# load and prepare data
filename = 'sonar.all-data.csv'
dataset = load_csv(filename)
for i in range(len(dataset[0])-1):
    str_column_to_float(dataset, i)
# convert string class to integers
str_column_to_int(dataset, len(dataset[0])-1)
# evaluate algorithm
n_folds = 3
l_rate = 0.01
n_epoch = 500
scores = evaluate_algorithm(dataset, perceptron, n_folds, l_rate, n_epoch)
print('Scores: %s' % scores)
print('Mean Accuracy: %.3f%%' % (sum(scores)/float(len(scores))))

## Q4: Diagram and forward pass for the perceptron

One unit with a Heaviside activation. All 60 inputs connect directly to it.

```
a = w^T x + b          w: 60 values, b: 1 value
y_hat = H(a)           H(a) = 1 if a >= 0 else 0
```

In [ ]:
from IPython.display import Image, display
display(Image("perceptron_architecture_diagram.png"))

## Q5 (extension): the same perceptron on FashionMNIST, top vs. trousers

The workbook's version of this cell kept the sonar loading code at the bottom, which overwrote the FashionMNIST dataset and raised `FileNotFoundError`. Here the sonar loading is removed, so the PCA output is what the perceptron trains on.

Not run here: it needs `fashion-mnist_train.csv` from Kaggle. Two things to know:
- It uses pure Python loops, so 500 epochs on about 12,000 images can take a while. I haven't timed it, so try a smaller `n_epoch` first.
- PCA is fit on all rows before the cross-validation split, so test folds influence the PCA directions. It doesn't use labels, but it is a small leak.

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA

# FashionMNIST label reference: 0=T-shirt/top, 1=Trouser, 2=Pullover, 3=Dress,
# 4=Coat, 5=Sandal, 6=Shirt, 7=Sneaker, 8=Bag, 9=Ankle boot
df = pd.read_csv('fashion-mnist_train.csv')

# Keep only two classes for binary classification: "top" (0) vs "pants" (1)
df_binary = df[df['label'].isin([0, 1])].reset_index(drop=True)

X_raw = df_binary.drop('label', axis=1).values          # raw 784-pixel vectors (28x28 flattened)
y_raw = df_binary['label'].values.astype(float)          # 0.0/1.0, the format the perceptron expects

n_components = 10
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_raw)

print(f"Reduced from {X_raw.shape[1]} pixels to {n_components} principal components")
print(f"Variance explained by {n_components} components: {sum(pca.explained_variance_ratio_):.3f}")

# Each row is [feature_1, ..., feature_10, label]
dataset = [list(X_pca[i]) + [y_raw[i]] for i in range(len(X_pca))]

seed(1)
n_folds = 3
l_rate = 0.01
n_epoch = 500
scores = evaluate_algorithm(dataset, perceptron, n_folds, l_rate, n_epoch)
print('Scores: %s' % scores)
print('Mean Accuracy: %.3f%%' % (sum(scores)/float(len(scores))))